# 🏦 Loan Eligibility Agent — LangGraph + RAG

**Full 5-Node Multi-Agent Workflow**

| Step | Cell(s) | What happens |
|------|---------|-------------|
| 0 | Install & Setup | Install all dependencies, set API key |
| 1 | Policy Document | Write `loan_rules.txt` with 16 rules |
| 2 | Build RAG | Chunk → Embed → FAISS index → `query_policies()` |
| 3 | LoanState | TypedDict shared across all nodes |
| 4 | Node 1 | `collect_financials` — LLM + regex extraction |
| 4 | Node 2 | `check_eligibility` — RAG + LLM policy check |
| 4 | Node 3 | `predict_approval_chance` — weighted math formula |
| 4 | Node 4 | `suggest_loan_plan` — EMI calculator |
| 4 | Node 5 | `review_response` — format + critic LLM pass |
| 5 | Build Graph | Wire nodes → compile LangGraph |
| 6 | Run Agent | Enter your query and get the full report |

> **Run cells top-to-bottom, one at a time.** Each cell is independent and labelled.

---
## ⚙️ Step 0A — Install Dependencies
Run this cell once. It installs LangChain, LangGraph, FAISS, and OpenAI.

In [ ]:
# Run once — installs all required packages
# This may take 2-3 minutes on first run

%pip install langchain langchain-community langchain-openai --quiet
%pip install langgraph --quiet
%pip install faiss-cpu --quiet
%pip install sentence-transformers --quiet
%pip install openai python-dotenv --quiet

print("✅ All packages installed successfully!")

---
## 🔑 Step 0B — Set Your OpenAI API Key

Replace `sk-...` with your actual key from https://platform.openai.com/api-keys

> **Free alternative:** See the comment block below to use HuggingFace embeddings + Ollama (no API key needed).

In [ ]:
import os

# ── Option A: OpenAI (recommended) ──────────────────────────────────
os.environ["OPENAI_API_KEY"] = "sk-YOUR-KEY-HERE"  # <-- replace this

# ── Option B: Load from a .env file (if you have one) ───────────────
# from dotenv import load_dotenv
# load_dotenv()   # reads OPENAI_API_KEY from .env file in same folder

# Verify the key is set
key = os.environ.get("OPENAI_API_KEY", "")
if key.startswith("sk-"):
    print(f"✅ API key loaded: sk-...{key[-6:]}")
else:
    print("⚠️  WARNING: API key not set or looks wrong. Update the cell above.")

---
## 📄 Step 1 — Create the Policy Document (`loan_rules.txt`)

**Why this file?**  
The RAG system in Node 2 will read this file, split it into chunks, embed each chunk as a vector, and store it in FAISS. When a customer query arrives, the system retrieves the most relevant rules based on semantic similarity — without hardcoding any if/else logic.

This file has **16 rules** across 8 sections — well above the 8-rule minimum.

In [ ]:
import os
from pathlib import Path

# Create the directory structure
policy_dir = Path("./data/bank_policies")
policy_dir.mkdir(parents=True, exist_ok=True)

policy_content = """
==========================================================
  NATIONAL LENDING BANK — LOAN ELIGIBILITY POLICY RULES
  Version 3.2 | Effective: January 2024
==========================================================

SECTION 1 — INCOME REQUIREMENTS
---------------------------------
Rule 1: Minimum Monthly Salary
  - Salaried employees: Rs.25,000 per month minimum net take-home pay.
  - Self-employed / business owners: Rs.35,000 per month minimum (average over last 3 months).
  - Freelancers / gig workers: Rs.40,000 per month minimum (average over last 6 months).
  - Government employees enjoy a 10% relaxation on the minimum income threshold.

Rule 2: Maximum Loan Amount Based on Salary
  - Salaried applicants: Maximum loan amount = 20x monthly net salary.
  - Self-employed: Maximum loan amount = 15x average monthly income.
  - No single loan shall exceed Rs.1,50,00,000 (1.5 crore) for personal/home loans.
  - Education loans are capped at Rs.75,00,000 regardless of income.

Rule 3: Income Proof Required
  - Last 3 months salary slips (salaried).
  - Last 2 years ITR with CA certification (self-employed).
  - Bank statements of the last 6 months are mandatory for all applicants.

SECTION 2 — CREDIT SCORE REQUIREMENTS
---------------------------------------
Rule 4: Minimum Credit Score Thresholds
  - Tier A (Preferred):  Credit score >= 750 -> Eligible for best interest rate band.
  - Tier B (Standard):   Credit score 700-749 -> Standard interest rate applies.
  - Tier C (Marginal):   Credit score 650-699 -> Approved with higher rate + collateral may be needed.
  - Tier D (High Risk):  Credit score 600-649 -> Conditional approval; guarantor required.
  - Below 600: Application automatically rejected. Applicant must wait 6 months before reapplying.

Rule 5: Credit Score Freshness
  - Credit score must have been pulled within the last 90 days.
  - Scores older than 90 days trigger a fresh bureau inquiry before processing.

SECTION 3 — DEBT-TO-INCOME (DTI) RATIO POLICY
------------------------------------------------
Rule 6: DTI Ratio Limits
  - Preferred DTI: <= 35% of monthly gross income.
  - Acceptable DTI: 36%-45% (loan processed with additional scrutiny).
  - Borderline DTI: 46%-55% (requires senior credit officer approval).
  - DTI > 55%: Application rejected. Applicant must reduce existing obligations.
  - DTI Calculation: (All existing EMIs + proposed new EMI) / Monthly gross income x 100.

Rule 7: EMI Burden Cap
  - After sanctioning the loan, total monthly EMI obligations must not exceed 50% of net monthly income.
  - If the proposed loan pushes total EMI above 50% of net income, the loan amount must be reduced.

SECTION 4 — INTEREST RATE SCHEDULE
-------------------------------------
Rule 8: Interest Rate Bands by Credit Tier
  - Tier A (score >= 750): 9.00% - 9.75% per annum (floating).
  - Tier B (700-749):     10.00% - 10.75% per annum.
  - Tier C (650-699):     11.00% - 12.00% per annum.
  - Tier D (600-649):     12.50% - 13.50% per annum.
  - Fixed rate option available only for Tier A and Tier B at 0.5% premium over floating.

Rule 9: Processing Fee Structure
  - Standard processing fee: 1% of sanctioned loan amount + 18% GST.
  - Minimum processing fee: Rs.5,000. Maximum processing fee: Rs.50,000.
  - Government employees: 50% waiver on processing fee.

SECTION 5 — LOAN TENURE POLICY
---------------------------------
Rule 10: Tenure Limits by Loan Type
  - Home Loan: 5-30 years (applicant age + tenure must not exceed 70 years).
  - Personal Loan: 1-7 years.
  - Car Loan: 1-8 years.
  - Education Loan: 1-15 years (moratorium of 1 year after course completion).
  - Business Loan: 1-10 years.

Rule 11: Age-Based Tenure Cap
  - Minimum applicant age: 21 years.
  - Maximum applicant age at loan maturity: 65 years (salaried), 70 years (self-employed).
  - For home loans, preferred tenure is 15-20 years for optimal EMI affordability.

SECTION 6 — EMPLOYMENT STABILITY REQUIREMENTS
-----------------------------------------------
Rule 12: Minimum Employment Tenure
  - Salaried (private sector): Minimum 1 year at current employer; 2 years total work experience.
  - Salaried (government/PSU): Minimum 6 months at current posting.
  - Self-employed: Business must be operational for at least 3 years with filed ITR.
  - Probationary employees are not eligible until confirmation letter is issued.

SECTION 7 — COLLATERAL AND GUARANTOR RULES
--------------------------------------------
Rule 13: Collateral Requirements
  - Personal loans above Rs.15 lakhs require property collateral or FD lien.
  - Home loans: The property being purchased serves as collateral (LTV <= 80%).
  - LTV (Loan-to-Value) ratio must not exceed 80% for home loans.

Rule 14: Guarantor Policy
  - Required when: Credit score is in Tier D (600-649), or DTI is between 46-55%.
  - Guarantor must have credit score >= 750 and monthly income >= 1.5x the borrower's income.

SECTION 8 — SPECIAL CATEGORIES
---------------------------------
Rule 15: Women Applicants
  - Additional 0.25% discount on interest rate for women as primary applicant.
  - Reduced processing fee: 0.75% instead of 1%.

Rule 16: NRI Applicants
  - NRI applicants must provide income proof in foreign currency at RBI reference rate.
  - Minimum NRI income equivalent: Rs.60,000/month.
  - NRI loans capped at Rs.50 lakhs for personal loans.

==========================================================
  END OF POLICY DOCUMENT
==========================================================
"""

policy_path = policy_dir / "loan_rules.txt"
policy_path.write_text(policy_content, encoding="utf-8")

print(f"✅ Policy document written to: {policy_path.resolve()}")
print(f"   Size: {policy_path.stat().st_size} bytes")
print(f"   Rules: 16 rules across 8 sections")

---
## 🔍 Step 2A — Build RAG: Load & Chunk the Policy Document

**What is chunking?**  
The full policy file is too large to embed as one vector — we'd lose precision. We split it into overlapping 500-character chunks. Each chunk gets its own embedding vector, so a question about "credit score" retrieves only the credit-score chunks, not the entire file.

- `chunk_size=500` → ~125 words per chunk  
- `chunk_overlap=80` → prevents rules from being split mid-sentence across chunk boundaries

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Load the policy document ─────────────────────────────────────────
# WHY TextLoader? It reads the .txt file and returns a LangChain
# Document object. For PDFs, swap to PyPDFLoader.

loader = TextLoader("./data/bank_policies/loan_rules.txt", encoding="utf-8")
documents = loader.load()
print(f"📄 Loaded {len(documents)} document(s)")

# ── Chunk the document ───────────────────────────────────────────────
# WHY RecursiveCharacterTextSplitter?
# It tries to split on paragraph breaks (\n\n) first, then newlines,
# then "Rule " markers, then spaces — preserving rule boundaries.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", "Rule ", "  -", " "],
)

chunks = splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks")
print()
print("── Preview of first 3 chunks ──")
for i, c in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} ({len(c.page_content)} chars):")
    print(c.page_content[:200], "...")

---
## 🧠 Step 2B — Build RAG: Embed Chunks & Create FAISS Vector Store

**What is embedding?**  
Each chunk is converted to a list of ~1536 numbers (a vector) that captures its *meaning*. Chunks about credit scores cluster near each other in this vector space. When a query arrives, we embed it too and find the closest chunks — that's semantic search.

> **Free alternative:** Comment out `OpenAIEmbeddings` and uncomment `HuggingFaceEmbeddings` below.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
# FREE ALTERNATIVE (no API key needed):
# from langchain_community.embeddings import HuggingFaceEmbeddings

print("🔄 Embedding chunks... (calls OpenAI API once per chunk)")

# ── Choose embedding model ────────────────────────────────────────────
# Option A: OpenAI (requires API key, best quality)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Option B: Free HuggingFace (no API key, slightly lower quality)
# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

# ── Build FAISS index from the chunks ────────────────────────────────
# .from_documents() calls the embedding API for each chunk,
# then builds an efficient nearest-neighbour index.
vectorstore = FAISS.from_documents(chunks, embeddings)

# ── Save to disk so we don't rebuild every time ───────────────────────
VECTORSTORE_PATH = "./vectorstore/faiss_index"
Path(VECTORSTORE_PATH).parent.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(VECTORSTORE_PATH)

print(f"✅ FAISS vector store saved to {VECTORSTORE_PATH}")
print(f"   Total vectors stored: {vectorstore.index.ntotal}")

---
## 🔎 Step 2C — Define `query_policies()` Function & Test It

This is the **core RAG function** used by Node 2. It takes a natural language question and returns the top-k most relevant policy chunks as a single string ready to inject into an LLM prompt.

In [ ]:
def query_policies(question: str, k: int = 4) -> str:
    """
    ✅ DELIVERABLE: RAG retriever function.

    Args:
        question: Natural language query about loan policies
        k: Number of chunks to retrieve (default 4)

    Returns:
        String of concatenated relevant policy chunks.
        Ready to inject directly into an LLM prompt.

    How it works:
        1. Embed the question into the same vector space as the policy chunks
        2. Find the k chunks with smallest cosine distance to the question
        3. Concatenate and return them

    Example:
        rules = query_policies("minimum credit score for home loan")
        # Returns text of Rule 4, Rule 5 etc.
    """
    retrieved_docs = vectorstore.similarity_search(question, k=k)
    combined = "\n\n---\n\n".join(
        [f"[Policy Chunk {i+1}]\n{doc.page_content}"
         for i, doc in enumerate(retrieved_docs)]
    )
    return combined


# ── Test the retriever with sample queries ────────────────────────────
print("=" * 55)
print("TEST 1: Query about credit score requirements")
print("=" * 55)
result1 = query_policies("minimum credit score required for loan approval", k=2)
print(result1)

print("\n" + "=" * 55)
print("TEST 2: Query about DTI ratio limits")
print("=" * 55)
result2 = query_policies("maximum DTI ratio allowed debt to income", k=2)
print(result2)

---
## 📋 Step 3 — Define `LoanState` (Shared State)

**Why a shared state?**  
LangGraph passes this dictionary between every node. Each node reads what it needs and writes back only the fields it populates. Think of it as the loan application file that travels from officer to officer.

- **Node 1** fills: `salary`, `credit_score`, `loan_amount`, `tenure_years`, `existing_emi`  
- **Node 2** fills: `retrieved_rules`, `eligibility_status`, `eligibility_reasons`  
- **Node 3** fills: `approval_score`, `score_breakdown`  
- **Node 4** fills: `suggested_plan`  
- **Node 5** fills: `final_reply`

In [ ]:
from typing import TypedDict, Optional, List

class LoanState(TypedDict):
    """
    ✅ DELIVERABLE: Shared state model for the LangGraph workflow.

    All fields start as None and get populated by the node
    responsible for extracting or computing them.

    WHY TypedDict?
    LangGraph merges partial dict updates between nodes.
    TypedDict gives us type hints without Pydantic overhead.
    Each node only returns the fields it changed.
    """
    # INPUT: raw user message
    user_query:          str

    # NODE 1 OUTPUTS — collect_financials
    salary:              Optional[float]   # monthly net salary in Rs
    credit_score:        Optional[int]     # CIBIL score (300-900)
    loan_amount:         Optional[float]   # total loan requested in Rs
    tenure_years:        Optional[int]     # preferred repayment period
    existing_emi:        Optional[float]   # current monthly EMI obligations
    loan_purpose:        Optional[str]     # home/personal/car/education/business
    employment_type:     Optional[str]     # salaried/self_employed/government/freelancer
    age:                 Optional[int]     # applicant age in years

    # NODE 2 OUTPUTS — check_eligibility (RAG)
    retrieved_rules:     Optional[str]     # policy chunks from FAISS
    eligibility_status:  Optional[str]     # ELIGIBLE / INELIGIBLE / CONDITIONAL
    eligibility_reasons: Optional[List]    # list of PASS/FAIL rule checks

    # NODE 3 OUTPUTS — predict_approval_chance
    approval_score:      Optional[float]   # 0-100 probability
    score_breakdown:     Optional[dict]    # per-factor score details

    # NODE 4 OUTPUTS — suggest_loan_plan
    suggested_plan:      Optional[dict]    # EMI, rate, tenure, processing fee

    # NODE 5 OUTPUTS — review_response
    final_reply:         Optional[str]     # customer-ready formatted report


def create_initial_state(user_query: str) -> LoanState:
    """Build a fresh state dict with all optional fields set to None."""
    return LoanState(
        user_query=user_query,
        salary=None, credit_score=None, loan_amount=None,
        tenure_years=None, existing_emi=None, loan_purpose=None,
        employment_type=None, age=None,
        retrieved_rules=None, eligibility_status=None, eligibility_reasons=None,
        approval_score=None, score_breakdown=None,
        suggested_plan=None, final_reply=None,
    )


# Quick smoke test
test_state = create_initial_state("I earn 50000 and need a 10 lakh loan")
print("✅ LoanState created successfully")
print(f"   Fields: {list(test_state.keys())}")
print(f"   user_query: '{test_state['user_query']}'")

---
## 🤖 Step 4A — Node 1: `collect_financials`

**Responsibility:** Parse the user's free-text chat message into structured numbers.

**Strategy:**  
1. LLM extracts all fields as JSON (handles natural language like "fifty thousand monthly")  
2. Regex fallback catches any field the LLM missed

**Why both LLM + regex?** The LLM handles ambiguous language; regex catches simple patterns if the LLM returns malformed JSON.

In [ ]:
import re
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Shared LLM instance — temperature=0 for deterministic outputs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=1000)


def collect_financials(state: LoanState) -> dict:
    """
    Node 1: Extract structured financial data from user's chat message.

    Returns dict with keys: salary, credit_score, loan_amount,
    tenure_years, existing_emi, loan_purpose, employment_type, age
    """
    print("[Node 1] collect_financials — extracting structured data...")
    user_query = state["user_query"]

    # ── LLM Extraction ──────────────────────────────────────────────
    extraction_prompt = f"""Extract financial information from the customer message.
Return ONLY valid JSON with these exact keys (use null if not mentioned):
{{
  "salary": <monthly net salary in rupees as number, or null>,
  "credit_score": <CIBIL score as integer 300-900, or null>,
  "loan_amount": <loan amount in rupees as number, or null>,
  "tenure_years": <tenure in years as integer, or null>,
  "existing_emi": <existing monthly EMI in rupees as number, or null>,
  "loan_purpose": <home/personal/car/education/business or null>,
  "employment_type": <salaried/self_employed/government/freelancer or null>,
  "age": <age in years as integer, or null>
}}
RULES: Convert amounts ("20 lakhs" -> 2000000, "50k" -> 50000).
"no EMI" or "no loans" -> existing_emi: 0. JSON ONLY, no extra text.

Customer: "{user_query}"""

    response = llm.invoke([HumanMessage(content=extraction_prompt)])
    raw = re.sub(r"```(?:json)?", "", response.content).strip("`  \n")

    try:
        extracted = json.loads(raw)
    except json.JSONDecodeError:
        print("  [Node 1] JSON parse failed — using regex fallback")
        extracted = {}

    # ── Regex Fallback ────────────────────────────────────────────────
    text = user_query.lower()

    def to_amount(s):
        if not s: return None
        if re.search(r"lakh|lac", s, re.I): return float(re.search(r"[\d.]+", s).group()) * 100000
        if re.search(r"crore", s, re.I):    return float(re.search(r"[\d.]+", s).group()) * 10000000
        if re.search(r"k\b", s, re.I):      return float(re.search(r"[\d.]+", s).group()) * 1000
        m = re.search(r"[\d,]+", s)
        return float(m.group().replace(",", "")) if m else None

    if extracted.get("salary") is None:
        m = re.search(r"(?:salary|earn|income|make)[^\d]*(\d[\d,\.]*\s*(?:k|lakh)?)", text)
        if m: extracted["salary"] = to_amount(m.group(1))

    if extracted.get("credit_score") is None:
        m = re.search(r"(?:credit|cibil)[^\d]*(\d{3})", text)
        if m: extracted["credit_score"] = int(m.group(1))

    if extracted.get("loan_amount") is None:
        m = re.search(r"(?:loan|need|want|borrow)[^\d]*([\d,\.]+\s*(?:lakh|lac|crore|k)?)", text)
        if m: extracted["loan_amount"] = to_amount(m.group(1))

    if extracted.get("tenure_years") is None:
        m = re.search(r"(\d+)\s*(?:year|yr)s?", text)
        if m: extracted["tenure_years"] = int(m.group(1))

    if extracted.get("existing_emi") is None:
        extracted["existing_emi"] = 0.0

    print(f"  Extracted: {json.dumps({k: v for k, v in extracted.items() if v is not None}, indent=4)}")

    return {
        "salary":          extracted.get("salary"),
        "credit_score":    extracted.get("credit_score"),
        "loan_amount":     extracted.get("loan_amount"),
        "tenure_years":    extracted.get("tenure_years"),
        "existing_emi":    extracted.get("existing_emi", 0.0),
        "loan_purpose":    extracted.get("loan_purpose"),
        "employment_type": extracted.get("employment_type"),
        "age":             extracted.get("age"),
    }


# ── Unit test Node 1 ─────────────────────────────────────────────────
test_query = "Hi, I earn Rs.75,000 per month. Credit score 740. Need a home loan of 40 lakhs for 20 years. My current EMI is Rs.5,000. I'm 30 years old."
test_state = create_initial_state(test_query)
result = collect_financials(test_state)
print("\n✅ Node 1 output:")
for k, v in result.items():
    if v is not None:
        print(f"   {k}: {v}")

---
## 🔎 Step 4B — Node 2: `check_eligibility` (RAG Node)

**Responsibility:** Retrieve the most relevant policy rules from FAISS, then use an LLM to apply those rules to the applicant's data.

**Why RAG here instead of hardcoded if/else?**  
When the bank updates its policy (e.g., raises minimum credit score from 700 to 720), you just update `loan_rules.txt` and rebuild the index — **zero code changes** needed.

In [ ]:
def check_eligibility(state: LoanState) -> dict:
    """
    Node 2: RAG-powered eligibility check.

    Steps:
    1. Compute DTI ratio from state data
    2. Build a targeted RAG query embedding the key metrics
    3. Retrieve top-5 most relevant policy chunks
    4. Pass chunks + applicant data to LLM for rule application
    5. Return eligibility_status + reasons list
    """
    print("[Node 2] check_eligibility — RAG query + policy check...")

    salary       = state.get("salary") or 0
    credit_score = state.get("credit_score") or 0
    loan_amount  = state.get("loan_amount") or 0
    tenure_years = state.get("tenure_years") or 15
    existing_emi = state.get("existing_emi") or 0
    purpose      = state.get("loan_purpose") or "personal"
    emp_type     = state.get("employment_type") or "salaried"
    age          = state.get("age") or 30

    # Compute DTI using standard reducing-balance EMI formula
    rate    = 0.09 if credit_score >= 750 else 0.10 if credit_score >= 700 else 0.11 if credit_score >= 650 else 0.125
    r, n    = rate / 12, tenure_years * 12
    p_emi   = loan_amount * (r * (1+r)**n) / ((1+r)**n - 1) if n > 0 else 0
    dti     = ((existing_emi + p_emi) / salary * 100) if salary > 0 else 999

    # Build a rich RAG query from computed metrics
    rag_query = (f"eligibility rules {purpose} loan credit score {credit_score} "
                 f"salary {salary} DTI {dti:.1f}% employment {emp_type} age {age}")
    retrieved_rules = query_policies(rag_query, k=5)
    print(f"  Retrieved {len(retrieved_rules.split('---'))} policy chunks from FAISS")

    # LLM applies the retrieved rules to applicant data
    prompt = f"""You are a strict bank credit officer. Apply these policy rules to the applicant.

=== POLICY RULES (from bank document) ===
{retrieved_rules}

=== APPLICANT DATA ===
Monthly salary:  Rs.{salary:,.0f} | Credit score: {credit_score}
Loan requested:  Rs.{loan_amount:,.0f} for {tenure_years} years ({purpose})
Existing EMI:    Rs.{existing_emi:,.0f}/month | Proposed EMI: Rs.{p_emi:,.0f}/month
DTI ratio:       {dti:.1f}% | Employment: {emp_type} | Age: {age}

Return ONLY valid JSON:
{{"eligibility_status": "ELIGIBLE"|"INELIGIBLE"|"CONDITIONAL",
  "reasons": ["PASS: rule description", "FAIL: rule description", ...]}}

ELIGIBLE=all hard rules pass. INELIGIBLE=at least one hard rule fails.
CONDITIONAL=passes but has borderline metrics needing scrutiny."""

    response = llm.invoke([HumanMessage(content=prompt)])
    raw = re.sub(r"```(?:json)?", "", response.content).strip("`  \n")

    try:
        result = json.loads(raw)
        status  = result.get("eligibility_status", "CONDITIONAL")
        reasons = result.get("reasons", [])
    except json.JSONDecodeError:
        # Deterministic fallback
        status  = "INELIGIBLE" if (credit_score < 600 or dti > 55 or salary < 25000) else \
                  "CONDITIONAL" if (credit_score < 700 or dti > 45) else "ELIGIBLE"
        reasons = [f"Credit: {credit_score}", f"DTI: {dti:.1f}%", f"Salary: Rs.{salary:,.0f}"]

    print(f"  Status: {status}")
    for r in reasons: print(f"  {r}")

    return {
        "retrieved_rules":     retrieved_rules,
        "eligibility_status":  status,
        "eligibility_reasons": reasons,
    }


# ── Unit test Node 2 ─────────────────────────────────────────────────
test_state2 = {**test_state, **result}   # merge Node 1 output into state
result2 = check_eligibility(test_state2)
print("\n✅ Node 2 output:")
print(f"   eligibility_status:  {result2['eligibility_status']}")
print(f"   reasons ({len(result2['eligibility_reasons'])} checks):")
for r in result2['eligibility_reasons']:
    print(f"     {r}")

---
## 📊 Step 4C — Node 3: `predict_approval_chance`

**Responsibility:** Compute a numerical approval probability using a weighted multi-factor formula.

**Why pure math — no LLM?**  
Scoring must be *deterministic* — same inputs must always produce same output. LLMs are non-deterministic even at `temperature=0`. Pure math is auditable, fast, and legally defensible.

| Factor | Weight | Why |
|--------|--------|-----|
| Credit score | 35% | Most predictive of default |
| DTI ratio | 25% | Measures cash flow risk |
| Income vs loan | 20% | Repayment capacity |
| Employment type | 12% | Stability of income |
| Tenure fit | 8% | Risk horizon match |

In [ ]:
def predict_approval_chance(state: LoanState) -> dict:
    """
    Node 3: Weighted scoring formula -> approval probability (0-100%).
    No LLM call — pure deterministic math.

    Returns: approval_score (float), score_breakdown (dict)
    """
    print("[Node 3] predict_approval_chance — computing weighted score...")

    salary       = state.get("salary") or 0
    credit_score = state.get("credit_score") or 0
    loan_amount  = state.get("loan_amount") or 0
    tenure_years = state.get("tenure_years") or 15
    existing_emi = state.get("existing_emi") or 0
    emp_type     = state.get("employment_type") or "salaried"

    # Recompute EMI and DTI
    rate  = 0.09 if credit_score >= 750 else 0.10 if credit_score >= 700 else 0.11 if credit_score >= 650 else 0.125
    r, n  = rate / 12, tenure_years * 12
    p_emi = loan_amount * (r * (1+r)**n) / ((1+r)**n - 1) if n > 0 else 0
    total_emi = existing_emi + p_emi
    dti   = (total_emi / salary * 100) if salary > 0 else 99

    # Factor 1: Credit Score (35%) — normalize 300-900 to 0-100
    f_credit = max(0, min(100, (credit_score - 300) / 600 * 100))

    # Factor 2: DTI Ratio (25%) — lower DTI = better score
    f_dti = max(0, 100 - (dti / 55 * 100))

    # Factor 3: Income vs Loan Ratio (20%) — fewer months of salary needed = better
    months_needed = loan_amount / salary if salary > 0 else 999
    f_income = max(0, 100 - (months_needed / 240 * 100))

    # Factor 4: Employment Stability (12%)
    f_emp = {"government": 100, "salaried": 85, "self_employed": 65, "freelancer": 45}.get(emp_type, 70)

    # Factor 5: Tenure Fit (8%) — optimal range is 10-20 years
    f_tenure = 100 if 10 <= tenure_years <= 20 else max(0, tenure_years/10*100) if tenure_years < 10 else max(0, 100-(tenure_years-20)*5)

    # Weighted sum
    score = round(f_credit*0.35 + f_dti*0.25 + f_income*0.20 + f_emp*0.12 + f_tenure*0.08, 1)

    breakdown = {
        "credit_score_factor":  round(f_credit, 1),
        "dti_ratio_factor":     round(f_dti, 1),
        "income_loan_factor":   round(f_income, 1),
        "employment_factor":    float(f_emp),
        "tenure_factor":        round(f_tenure, 1),
        "dti_ratio_computed":   round(dti, 1),
        "proposed_emi":         round(p_emi, 0),
        "total_emi_after_loan": round(total_emi, 0),
    }

    print(f"  Score: {score}%")
    print(f"  Credit={f_credit:.0f}  DTI={f_dti:.0f}  Income={f_income:.0f}  Emp={f_emp}  Tenure={f_tenure:.0f}")
    return {"approval_score": score, "score_breakdown": breakdown}


# ── Unit test Node 3 ─────────────────────────────────────────────────
test_state3 = {**test_state2, **result2}
result3 = predict_approval_chance(test_state3)
print(f"\n✅ Node 3 output:")
print(f"   approval_score: {result3['approval_score']}%")
print(f"   breakdown: {json.dumps(result3['score_breakdown'], indent=6)}")

---
## 💰 Step 4D — Node 4: `suggest_loan_plan`

**Responsibility:** Calculate a concrete loan offer — eligible amount, interest rate tier, monthly EMI, and total repayment — using the standard reducing-balance EMI formula.

**EMI Formula:**
$$EMI = \frac{P \times r \times (1+r)^n}{(1+r)^n - 1}$$

Where: P = principal, r = monthly interest rate, n = number of months

In [ ]:
def suggest_loan_plan(state: LoanState) -> dict:
    """
    Node 4: Compute an optimised loan plan using the EMI formula.

    Also computes alternate tenures (10/15/20 year comparison)
    so the customer can make an informed choice.

    Returns: suggested_plan (dict with full loan details)
    """
    print("[Node 4] suggest_loan_plan — computing optimal loan plan...")

    salary        = state.get("salary") or 0
    credit_score  = state.get("credit_score") or 0
    loan_amount   = state.get("loan_amount") or 0
    tenure_years  = state.get("tenure_years") or 15
    existing_emi  = state.get("existing_emi") or 0
    approval_score= state.get("approval_score") or 50

    # ── Determine interest rate band from credit score ───────────────
    if credit_score >= 750:   rate, tier, rate_str = 0.09,  "Tier A (Preferred)", "9.00%"
    elif credit_score >= 700: rate, tier, rate_str = 0.10,  "Tier B (Standard)",  "10.00%"
    elif credit_score >= 650: rate, tier, rate_str = 0.11,  "Tier C (Marginal)",  "11.00%"
    else:                     rate, tier, rate_str = 0.125, "Tier D (High Risk)", "12.50%"

    # ── Cap loan based on policy Rule 2 (max 20x salary) ────────────
    max_by_salary  = salary * 20
    score_cap      = 1.0 if approval_score >= 80 else 0.9 if approval_score >= 65 else 0.75 if approval_score >= 50 else 0.6
    recommended    = min(loan_amount, max_by_salary, loan_amount * score_cap)

    # ── Standard reducing-balance EMI formula ────────────────────────
    def emi_calc(P, annual_rate, years):
        mr = annual_rate / 12
        mn = years * 12
        return P * (mr * (1+mr)**mn) / ((1+mr)**mn - 1)

    emi   = emi_calc(recommended, rate, tenure_years)
    total = emi * tenure_years * 12
    interest_paid = total - recommended

    # ── Alternate tenures for comparison ─────────────────────────────
    alt_plans = []
    for t in [10, 15, 20]:
        if t != tenure_years:
            e = emi_calc(recommended, rate, t)
            alt_plans.append({
                "tenure_years":   t,
                "monthly_emi":    round(e, 0),
                "total_interest": round(e * t * 12 - recommended, 0),
            })

    plan = {
        "recommended_amount":  round(recommended, 0),
        "interest_rate":       rate_str,
        "rate_tier":           tier,
        "tenure_years":        tenure_years,
        "monthly_emi":         round(emi, 0),
        "total_interest_paid": round(interest_paid, 0),
        "total_repayment":     round(total, 0),
        "processing_fee":      round(min(50000, max(5000, recommended * 0.01)), 0),
        "alternate_plans":     alt_plans,
    }

    print(f"  Recommended: Rs.{recommended/100000:.1f}L @ {rate_str} -> EMI Rs.{emi:,.0f}/mo")
    return {"suggested_plan": plan}


# ── Unit test Node 4 ─────────────────────────────────────────────────
test_state4 = {**test_state3, **result3}
result4 = suggest_loan_plan(test_state4)
p = result4["suggested_plan"]
print(f"\n✅ Node 4 output:")
print(f"   Loan:     Rs.{p['recommended_amount']:,.0f}")
print(f"   Rate:     {p['interest_rate']} ({p['rate_tier']})")
print(f"   EMI:      Rs.{p['monthly_emi']:,.0f}/month")
print(f"   Total:    Rs.{p['total_repayment']:,.0f}")
print(f"   Alternate plans:")
for alt in p['alternate_plans']:
    print(f"     {alt['tenure_years']}yr: Rs.{alt['monthly_emi']:,.0f}/mo | Interest Rs.{alt['total_interest']:,.0f}")

---
## ✅ Step 4E — Node 5: `review_response`

**Responsibility:** Assemble all state fields into a structured, customer-ready report. Then pass the draft through an LLM critic that adds personalization and catches any inconsistencies.

**Why a separate review node?**  
This is the "critic pattern" in agentic AI. Node 4 generates content; Node 5 audits and polishes it. Separating generation from review prevents the system from sending inconsistent or incorrect advice to the customer.

In [ ]:
def review_response(state: LoanState) -> dict:
    """
    Node 5: Format the complete report + LLM critic pass.

    Steps:
    1. Build a structured ASCII report from all state fields
    2. Pass the draft to the LLM to add personalization + catch errors
    3. Return the polished final_reply string
    """
    print("[Node 5] review_response — formatting and reviewing final reply...")

    eligibility  = state.get("eligibility_status") or "CONDITIONAL"
    reasons      = state.get("eligibility_reasons") or []
    score        = state.get("approval_score") or 0
    plan         = state.get("suggested_plan") or {}
    breakdown    = state.get("score_breakdown") or {}
    salary       = state.get("salary") or 0
    credit_score = state.get("credit_score") or 0
    purpose      = state.get("loan_purpose") or "loan"

    # ── ASCII score bar ───────────────────────────────────────────────
    filled   = int(score / 5)
    score_bar = "█" * filled + "░" * (20 - filled)

    # ── Decision header ───────────────────────────────────────────────
    if eligibility == "ELIGIBLE":
        header = f"DECISION: APPROVED  (Approval likelihood: {score:.0f}%)"
    elif eligibility == "CONDITIONAL":
        header = f"DECISION: CONDITIONAL REVIEW  (Approval likelihood: {score:.0f}%)"
    else:
        header = f"DECISION: NOT ELIGIBLE  (Approval likelihood: {score:.0f}%)"

    # ── Alternate plan section ────────────────────────────────────────
    alt_section = ""
    if plan.get("alternate_plans"):
        alt_section = "\nALTERNATE TENURE OPTIONS\n" + "─" * 26
        for alt in plan["alternate_plans"]:
            alt_section += f"\n  {alt['tenure_years']}-year plan:  Rs.{alt['monthly_emi']:,.0f}/mo  |  Total interest Rs.{alt['total_interest']:,.0f}"

    # ── Build the draft report ────────────────────────────────────────
    draft = f"""
========================================================
  NATIONAL LENDING BANK — LOAN ASSESSMENT REPORT
========================================================

{header}
[{score_bar}] {score:.0f}/100

POLICY CHECK RESULTS
────────────────────
{chr(10).join(f'  * {r}' for r in reasons)}

RECOMMENDED LOAN PLAN
─────────────────────
  Loan Amount:     Rs.{plan.get('recommended_amount', 0):,.0f}
  Interest Rate:   {plan.get('interest_rate', 'N/A')}  ({plan.get('rate_tier', '')})
  Tenure:          {plan.get('tenure_years', 0)} years
  Monthly EMI:     Rs.{plan.get('monthly_emi', 0):,.0f}
  Total Interest:  Rs.{plan.get('total_interest_paid', 0):,.0f}
  Total Repayment: Rs.{plan.get('total_repayment', 0):,.0f}
  Processing Fee:  Rs.{plan.get('processing_fee', 0):,.0f} (+ 18% GST)
{alt_section}

SCORE BREAKDOWN
───────────────
  Credit score factor (35%):  {breakdown.get('credit_score_factor', 0):.0f}/100
  DTI ratio factor   (25%):  {breakdown.get('dti_ratio_factor', 0):.0f}/100
  Income/loan factor (20%):  {breakdown.get('income_loan_factor', 0):.0f}/100
  Employment factor  (12%):  {breakdown.get('employment_factor', 0):.0f}/100
  Tenure factor       (8%):  {breakdown.get('tenure_factor', 0):.0f}/100
  Computed DTI:               {breakdown.get('dti_ratio_computed', 0):.1f}%
  Proposed EMI:               Rs.{breakdown.get('proposed_emi', 0):,.0f}/month

NEXT STEPS
──────────
  * Submit application at nearest branch or via our mobile app.
  * Required: 3 months salary slips, 6 months bank statements, PAN card.
  * Processing time: 3-5 working days after document verification.
  * Helpline: 1800-XXX-XXXX (toll-free) | loans@nlb.in
========================================================
"""

    # ── LLM Critic: personalise and check consistency ─────────────────
    critic_prompt = f"""You are a senior bank manager reviewing this loan report.
1. Check for inconsistencies or errors in the numbers.
2. Add ONE personalised tip at the end based on the applicant's profile.
3. Keep the full report structure intact — only improve it.
4. Return the complete final report text.

Report:
{draft}

Profile: credit={credit_score}, salary=Rs.{salary:,.0f}, purpose={purpose}, score={score:.0f}%, status={eligibility}"""

    response = llm.invoke([HumanMessage(content=critic_prompt)])
    final_text = response.content.strip()

    print(f"  Final reply ready ({len(final_text)} chars)")
    return {"final_reply": final_text}


# ── Unit test Node 5 ─────────────────────────────────────────────────
test_state5 = {**test_state4, **result4}
result5 = review_response(test_state5)
print("\n" + "=" * 60)
print(result5["final_reply"])
print("=" * 60)

---
## 🔗 Step 5 — Build the LangGraph Workflow

**What is `StateGraph`?**  
It manages node registration, edge definition, and state merging. When you call `.compile()`, it validates the graph (no dangling edges, entry point set) and returns an `app` object with a `.invoke()` method.

**Why linear edges here?**  
Every application flows through all 5 nodes. The LLM at Node 5 formats a rejection message for ineligible applicants — the graph never silently skips a node. This makes the output consistent and debuggable.

In [ ]:
from langgraph.graph import StateGraph, END

def build_loan_graph():
    """
    Step 5: Compile the full 5-node LangGraph workflow.

    Graph edges:
        START
          -> collect_financials      (Node 1: LLM + regex extraction)
          -> check_eligibility       (Node 2: RAG + policy LLM)
          -> predict_approval_chance (Node 3: pure math)
          -> suggest_loan_plan       (Node 4: EMI formula)
          -> review_response         (Node 5: format + critic LLM)
          -> END
    """

    # 1. Create graph with LoanState as the shared state schema
    graph = StateGraph(LoanState)

    # 2. Register all 5 nodes
    graph.add_node("collect_financials",      collect_financials)
    graph.add_node("check_eligibility",       check_eligibility)
    graph.add_node("predict_approval_chance", predict_approval_chance)
    graph.add_node("suggest_loan_plan",       suggest_loan_plan)
    graph.add_node("review_response",         review_response)

    # 3. Set the entry point (first node to execute)
    graph.set_entry_point("collect_financials")

    # 4. Define execution order with directed edges
    graph.add_edge("collect_financials",      "check_eligibility")
    graph.add_edge("check_eligibility",       "predict_approval_chance")
    graph.add_edge("predict_approval_chance", "suggest_loan_plan")
    graph.add_edge("suggest_loan_plan",       "review_response")
    graph.add_edge("review_response",          END)

    # 5. Compile -> validates the graph and returns an executable runnable
    app = graph.compile()
    return app


loan_app = build_loan_graph()

print("✅ LangGraph workflow compiled successfully!")
print()
print("Execution order:")
print("  START")
print("    -> [Node 1] collect_financials      (LLM + regex)")
print("    -> [Node 2] check_eligibility       (RAG + LLM)")
print("    -> [Node 3] predict_approval_chance (Math only)")
print("    -> [Node 4] suggest_loan_plan       (EMI formula)")
print("    -> [Node 5] review_response         (Format + LLM critic)")
print("    -> END")

---
## 🚀 Step 6 — Run the Agent!

Enter your loan query in natural language. The agent will run all 5 nodes and print the full assessment report.

**Try these sample queries:**
- `"I earn Rs.75,000/month, credit score 740, need 40 lakh home loan for 20 years. EMI currently Rs.5,000. Age 30."`
- `"Salary 35000, cibil 680, personal loan 5 lakh for 3 years, no existing loans"`
- `"I make 22000 monthly, credit score 580, need 15 lakh loan"`  ← expect rejection

In [ ]:
# ── Enter your query here ─────────────────────────────────────────────
USER_QUERY = "I earn Rs.75,000 per month as a salaried software engineer. \
My CIBIL score is 740. I want a home loan of Rs.40 lakhs for 20 years. \
I currently pay Rs.5,000 EMI on a car loan. I am 30 years old."

# ── Run the full 5-node pipeline ──────────────────────────────────────
print("=" * 60)
print("RUNNING LOAN ELIGIBILITY AGENT")
print("=" * 60)
print(f"Query: {USER_QUERY}")
print()

initial_state = create_initial_state(USER_QUERY)
final_state   = loan_app.invoke(initial_state)

print("\n" + "=" * 60)
print("FINAL CUSTOMER REPORT")
print("=" * 60)
print(final_state.get("final_reply", "No reply generated."))

---
## 🎯 Bonus — Test Multiple Scenarios

Run different applicant profiles through the pipeline and compare results.

In [ ]:
test_cases = [
    {
        "label": "Strong Applicant (expect APPROVED)",
        "query": "I earn Rs.1,00,000/month, credit score 780, need 50 lakh home loan for 15 years. No existing loans. Age 35, salaried MNC."
    },
    {
        "label": "Borderline Applicant (expect CONDITIONAL)",
        "query": "My salary is Rs.40,000. Credit score is 670. I need a personal loan of Rs.8 lakhs for 5 years. Current EMI is Rs.8,000."
    },
    {
        "label": "Weak Applicant (expect INELIGIBLE)",
        "query": "I earn 20,000 per month. My credit score is 570. I want a loan of 15 lakhs."
    },
]

for tc in test_cases:
    print(f"\n{'='*60}")
    print(f"TEST: {tc['label']}")
    print(f"{'='*60}")

    state  = create_initial_state(tc["query"])
    result = loan_app.invoke(state)

    # Show just the summary line
    reply = result.get("final_reply", "")
    for line in reply.split("\n"):
        if "DECISION" in line or "Approval" in line or "likelihood" in line:
            print(f"  {line.strip()}")
    print(f"  Approval Score: {result.get('approval_score', 0)}%")
    print(f"  Status: {result.get('eligibility_status', 'N/A')}")

print("\n✅ All test cases completed!")

---
## 📚 Quick Reference

| File / Object | What it is |
|---|---|
| `data/bank_policies/loan_rules.txt` | 16 policy rules (Step 1 deliverable) |
| `query_policies(q, k)` | RAG retriever — returns top-k policy chunks (Step 2 deliverable) |
| `LoanState` | TypedDict shared across all 5 nodes (Step 3 deliverable) |
| `collect_financials` | Node 1 — LLM + regex extraction |
| `check_eligibility` | Node 2 — RAG + LLM policy check |
| `predict_approval_chance` | Node 3 — Weighted math formula |
| `suggest_loan_plan` | Node 4 — EMI calculator + plan |
| `review_response` | Node 5 — Format report + LLM critic |
| `loan_app` | Compiled `StateGraph` — call `.invoke(state)` |

**To update policies:** Edit `loan_rules.txt` and re-run Step 2B cells. No code changes needed.

**To use free HuggingFace embeddings:** In Step 2B, swap `OpenAIEmbeddings` for `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")`.